# GTF-based end-to-end subisoform simulation

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "tealeaf").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)


In [2]:
from pathlib import Path

import pandas as pd

from tealeaf.subisoform_simulation import (
    build_gtf_event_simulation_design,
    run_end_to_end_simulation,
)


In [3]:
gtf_path = Path("/gpfs/commons/groups/knowles_lab/index/kallisto/mus_musculus/with_precursor/gencode.vM32.basic.annotation.gtf")
transcript_ids = ["ENSMUST00000204394.3", "ENSMUST00000204423.3"]
design = build_gtf_event_simulation_design(gtf_path, transcript_ids=transcript_ids)

design.subisoform_model.subisoform_table.loc[
    design.subisoform_model.subisoform_table.event_id == design.target_event_id,
    ["event_id", "subisoform_id", "transcript_ids", "segment_ids"],
]

,event_id,subisoform_id,transcript_ids,segment_ids
2,ENSMUSG00000107872.3:evt002,ENSMUSG00000107872.3:sub002,"(ENSMUST00000204394.3,)","(ENSMUSG00000107872.3:seg005, ENSMUSG000001078..."
3,ENSMUSG00000107872.3:evt002,ENSMUSG00000107872.3:sub003,"(ENSMUST00000204423.3,)","(ENSMUSG00000107872.3:seg005, ENSMUSG000001078..."


In [4]:
simulation, pipeline = run_end_to_end_simulation(
    design=design,
    alpha_by_condition={"a": [18, 2], "b": [2, 18]},
    n_cells_per_condition=18,
    mean_gene_count=120,
    metacell_size=3,
    random_state=0,
)

pd.DataFrame(
    {
        "condition": simulation.group_labels,
        "total_ec_count": simulation.cell_ec_matrix.sum(axis=1).A1,
    }
).groupby("condition").agg(["size", "mean"])

total_ec_count            
                    size        mean
condition                           
a                     18  121.500000
b                     18  118.888889

In [5]:
pipeline.differential_usage.loc[:, ["event_id", "n_subisoforms", "n_samples", "lrt_statistic", "p_value", "fdr"]]

,event_id,n_subisoforms,n_samples,lrt_statistic,p_value,fdr
0,ENSMUSG00000107872.3:evt002,2,12,37.332113,7.824104e-09,7.824104e-09
